# D and f4 statistics from an alignment: L42 example

This notebook retains the final L42 implementation as one representative matrix example. The same workflow was applied to other matrices by changing the input alignment, species tree, clade map, and output directory.

It is provided for methodological transparency rather than as a turnkey reproducibility workflow. Set the `ANALYSIS_ROOT` environment variable before adapting it to another directory structure. Figure-generation and exploratory display cells have been removed.


In [ ]:
# ABBA–BABA (D) and f4 on biallelic amino-acid sites with quartet generation,
# orientation of (P1,P2) as sisters, and resumable, per-quartet appending to a TSV.

import os
from pathlib import Path
from math import sqrt, erfc
from typing import Dict, List, Tuple, Set
from collections import defaultdict
import sys

import pandas as pd  # pip install pandas
try:
    from Bio import Phylo  # pip install biopython
except ImportError:
    sys.exit("This script requires Biopython. Install with: pip install biopython")

# Optional t-based p-values
try:
    from scipy.stats import t as t_dist  # pip install scipy
except Exception:
    t_dist = None

# Optional Benjamini-Hochberg correction from statsmodels
try:
    from statsmodels.stats.multitest import multipletests
except Exception:
    multipletests = None


# -------------------- USER INPUTS --------------------
# These paths refer to the original analysis workspace, not the GitHub repository.
# Set ANALYSIS_ROOT in the environment before adapting this example.
ANALYSIS_ROOT = Path(os.environ.get("ANALYSIS_ROOT", "/path/to/original_analysis_workspace"))
DATA_DIR = ANALYSIS_ROOT / "data"
SCRATCH_DIR = ANALYSIS_ROOT / "scratch"

alignment_path = str(DATA_DIR / "gene_alignments/L42_synt.concatenated.fa")
tree_path = str(DATA_DIR / "trees/L42.mycladetopo.topo29.with_node_labels.tree")
clade_map_path = str(DATA_DIR / "trees/L42_clades.csv")
clades_of_interest = [
    "Owenidae","Chaetopteridae","Sipuncula","Amphinomida","Spiosabelliformia",
    "Capitellida","Terebellida","Clitellata","Cirratuliformia","Siboglinidae","Errantia"
]
output_dir = str(SCRATCH_DIR / "tree_building/ILS/L42_synt/all_clades/ABBA_BABA")
output_file = os.path.join(output_dir, "abba_baba_f4_crossclade_incremental.tsv")

# Outgroup: specify exactly one of these
outgroup_taxon = "LILO"
outgroup_clade = "Nemertea"  # ignored if outgroup_taxon is set

# Quartet generation options
require_P3_third_clade = True
use_all_outgroup_taxa = True
max_p1_per_clade = 2
max_p2_per_clade = 2
max_p3_per_clade = 2
max_pairs_cross_clades = None

# Jackknife
block_size = 100
# -----------------------------------------------------

# Amino-acid alphabets
VALID_AA = set("ACDEFGHIKLMNPQRSTVWYUO")  # include U,O
MISSING = set(["-", ".", "X", "?", "B", "Z", "J"])  # gaps/ambig => missing

SAFE_MIN_P = 1e-300


# -------------------- HELPERS --------------------
def parse_fasta(path: str) -> Dict[str, str]:
    seqs = {}
    with open(path) as fh:
        name = None
        chunks = []
        for line in fh:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if name is not None:
                    seqs[name] = "".join(chunks).upper()
                name = line[1:].split()[0]
                chunks = []
            else:
                chunks.append(line)
        if name is not None:
            seqs[name] = "".join(chunks).upper()
    if not seqs:
        raise ValueError(f"No sequences found in {path}")
    lens = {len(s) for s in seqs.values()}
    if len(lens) != 1:
        raise ValueError("Alignment is not rectangular: sequence lengths differ.")
    return seqs


def parse_clade_map(path: str) -> Dict[str, Set[str]]:
    clade2taxa = defaultdict(set)
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) < 2:
                continue
            clade = parts[0]
            taxa = parts[1].split(",")
            for t in taxa:
                t = t.strip()
                if t:
                    clade2taxa[clade].add(t)
    return dict(clade2taxa)


def aa_is_valid(c: str) -> bool:
    return c in VALID_AA


def site_is_biallelic(a1: str, a2: str, a3: str, a4: str) -> Tuple[bool, List[str]]:
    alleles = {a1, a2, a3, a4}
    if len(alleles) != 2:
        return False, []
    return True, sorted(list(alleles))


def norm_twosided_p_from_Z(Z: float) -> float:
    if not (Z == Z):  # NaN
        return float("nan")
    p = 2.0 * 0.5 * erfc(abs(Z) / sqrt(2.0))
    if p != p:
        return float("nan")
    return max(min(p, 1.0), SAFE_MIN_P)


def compute_D_and_f4_for_quartet(seqs: Dict[str, str], p1: str, p2: str, p3: str, o: str, block_size: int):
    s1, s2, s3, so = seqs[p1], seqs[p2], seqs[p3], seqs[o]
    L = len(s1)

    ABBA_sites = [0] * L
    BABA_sites = [0] * L
    f4_sites = [None] * L

    n_D_sites = 0
    n_f4_sites = 0

    for i in range(L):
        a1, a2, a3, ao = s1[i], s2[i], s3[i], so[i]
        if (a1 in MISSING or a2 in MISSING or a3 in MISSING or ao in MISSING):
            continue
        if not (aa_is_valid(a1) and aa_is_valid(a2) and aa_is_valid(a3) and aa_is_valid(ao)):
            continue

        is_bi, alleles = site_is_biallelic(a1, a2, a3, ao)
        if not is_bi:
            continue

        # f4 coding (alleles[1] as '1')
        allele_b = alleles[1]
        x1 = 1 if a1 == allele_b else 0
        x2 = 1 if a2 == allele_b else 0
        x3 = 1 if a3 == allele_b else 0
        xO = 1 if ao == allele_b else 0
        f4_sites[i] = (x1 - x2) * (x3 - xO)
        n_f4_sites += 1

        # D: polarize by outgroup allele
        A = ao
        B = alleles[0] if alleles[1] == ao else alleles[1]
        if a1 == A and a2 == B and a3 == B:
            ABBA_sites[i] = 1
            n_D_sites += 1
        elif a1 == B and a2 == A and a3 == B:
            BABA_sites[i] = 1
            n_D_sites += 1

    total_ABBA = sum(ABBA_sites)
    total_BABA = sum(BABA_sites)
    D_denom = total_ABBA + total_BABA
    D = (total_ABBA - total_BABA) / D_denom if D_denom > 0 else float("nan")

    f4_sum = sum(v for v in f4_sites if v is not None)
    f4_mean = (f4_sum / n_f4_sites) if n_f4_sites > 0 else float("nan")

    def jackknife_D():
        if D_denom == 0:
            return float("nan"), float("nan"), 0
        starts = list(range(0, L, block_size))
        Bn = len(starts)
        if Bn <= 1:
            return float("nan"), float("nan"), Bn
        blocks_ABBA = []
        blocks_BABA = []
        for start in starts:
            end = min(start + block_size, L)
            blocks_ABBA.append(sum(ABBA_sites[start:end]))
            blocks_BABA.append(sum(BABA_sites[start:end]))
        D_lo = []
        for i_blk in range(Bn):
            abba = total_ABBA - blocks_ABBA[i_blk]
            baba = total_BABA - blocks_BABA[i_blk]
            denom = abba + baba
            D_lo.append((abba - baba) / denom if denom > 0 else (total_ABBA - total_BABA) / (D_denom if D_denom > 0 else 1.0))
        mean_lo = sum(D_lo) / Bn
        var = (Bn - 1) / Bn * sum((d - mean_lo) ** 2 for d in D_lo)
        se = sqrt(var)
        Z = (D / se) if se and se == se else float("nan")
        return se, Z, Bn

    def jackknife_f4():
        if n_f4_sites == 0:
            return float("nan"), float("nan"), 0
        starts = list(range(0, L, block_size))
        Bn = len(starts)
        if Bn <= 1:
            return float("nan"), float("nan"), Bn
        blocks_sum = []
        blocks_n = []
        for start in starts:
            end = min(start + block_size, L)
            vals = [v for v in f4_sites[start:end] if v is not None]
            blocks_sum.append(sum(vals))
            blocks_n.append(len(vals))
        total_sum = f4_sum
        total_n = n_f4_sites
        f4_lo = []
        for i_blk in range(Bn):
            s = total_sum - blocks_sum[i_blk]
            n = total_n - blocks_n[i_blk]
            f4_lo.append((s / n) if n > 0 else f4_mean)
        mean_lo = sum(f4_lo) / Bn
        var = (Bn - 1) / Bn * sum((x - mean_lo) ** 2 for x in f4_lo)
        se = sqrt(var)
        Z = (f4_mean / se) if se and se == se else float("nan")
        return se, Z, Bn

    D_se, D_Z, D_B = jackknife_D()
    f4_se, f4_Z, f4_B = jackknife_f4()

    # Normal-approx p-values
    p_D_norm = norm_twosided_p_from_Z(D_Z)
    p_f4_norm = norm_twosided_p_from_Z(f4_Z)

    # t-based p-values with df = B-1 if SciPy available; else normal
    if t_dist is not None and D_B > 1 and D_Z == D_Z:
        p_D_t = max(min(2.0 * t_dist.sf(abs(D_Z), df=D_B - 1), 1.0), SAFE_MIN_P)
        p_D_method = "t"
    else:
        p_D_t = p_D_norm
        p_D_method = "normal"

    if t_dist is not None and f4_B > 1 and f4_Z == f4_Z:
        p_f4_t = max(min(2.0 * t_dist.sf(abs(f4_Z), df=f4_B - 1), 1.0), SAFE_MIN_P)
        p_f4_method = "t"
    else:
        p_f4_t = p_f4_norm
        p_f4_method = "normal"

    return {
        "sites_D": int(n_D_sites),
        "ABBA": int(total_ABBA),
        "BABA": int(total_BABA),
        "D": D,
        "D_SE": D_se,
        "D_Z": D_Z,
        "D_blocks": int(D_B),
        "p_D_norm": p_D_norm,
        "p_D_t": p_D_t,
        "p_D_method": p_D_method,
        "sites_f4": int(n_f4_sites),
        "f4_mean": f4_mean,
        "f4_SE": f4_se,
        "f4_Z": f4_Z,
        "f4_blocks": int(f4_B),
        "p_f4_norm": p_f4_norm,
        "p_f4_t": p_f4_t,
        "p_f4_method": p_f4_method,
    }


def generate_quartets_cross_clades(
    seqs: Dict[str, str],
    tree,
    clade2taxa: Dict[str, Set[str]],
    interest: List[str],
    out_taxon: str = None,
    out_clade: str = None,
    require_P3_third: bool = True,
    use_all_out: bool = True,
    max_p1: int = None,
    max_p2: int = None,
    max_p3: int = None,
    max_pairs_cross: int = None,
) -> List[Dict[str, str]]:
    present_alignment_taxa = set(seqs.keys())
    tree_taxa = {t.name for t in tree.get_terminals()}
    present_taxa = present_alignment_taxa & tree_taxa

    # Outgroup taxa
    if out_taxon:
        if out_taxon not in present_taxa:
            raise ValueError(f"Outgroup taxon '{out_taxon}' not present in both alignment and tree.")
        outgroup_taxa = [out_taxon]
        outgroup_label = "OUT"
    else:
        if not out_clade:
            raise ValueError("Provide either outgroup_taxon or outgroup_clade.")
        if out_clade not in clade2taxa:
            raise ValueError(f"Outgroup clade '{out_clade}' not found in clade map.")
        out_all = sorted(clade2taxa[out_clade] & present_taxa)
        if not out_all:
            raise ValueError(f"No outgroup taxa from clade '{out_clade}' found in both alignment and tree.")
        outgroup_taxa = out_all if use_all_out else [out_all[0]]
        outgroup_label = out_clade

    # Filter clade taxa to those present
    clade2taxa_present = {c: sorted(list(tset & present_taxa)) for c, tset in clade2taxa.items()}
    interest = [c for c in interest if clade2taxa_present.get(c)]

    quartets = []
    for i, c1 in enumerate(interest):
        taxa_c1 = clade2taxa_present[c1]
        if len(taxa_c1) == 0:
            continue
        p1_candidates = taxa_c1[:max_p1] if (max_p1 is not None) else taxa_c1

        for j in range(i + 1, len(interest)):
            c2 = interest[j]
            taxa_c2 = clade2taxa_present[c2]
            if len(taxa_c2) == 0:
                continue
            p2_candidates = taxa_c2[:max_p2] if (max_p2 is not None) else taxa_c2

            pair_count = 0
            for p1 in p1_candidates:
                for p2 in p2_candidates:
                    if max_pairs_cross is not None and pair_count >= max_pairs_cross:
                        break
                    for c3 in interest:
                        if require_P3_third and (c3 == c1 or c3 == c2):
                            continue
                        taxa_c3 = clade2taxa_present[c3]
                        if len(taxa_c3) == 0:
                            continue
                        p3_list = taxa_c3[:max_p3] if (max_p3 is not None) else taxa_c3
                        for p3 in p3_list:
                            for o in outgroup_taxa:
                                quartets.append({
                                    "P1": p1, "P2": p2, "P3": p3, "O": o,
                                    "P1_clade": c1, "P2_clade": c2, "P3_clade": c3, "O_clade": outgroup_label
                                })
                            pair_count += 1
                    if max_pairs_cross is not None and pair_count >= max_pairs_cross:
                        break
    return quartets


def sister_pair_for_triplet(tree, a, b, c):
    ca_ab = tree.common_ancestor(a, b)
    ca_ac = tree.common_ancestor(a, c)
    ca_bc = tree.common_ancestor(b, c)
    ab_has_c = any(t.name == c for t in ca_ab.get_terminals())
    ac_has_b = any(t.name == b for t in ca_ac.get_terminals())
    bc_has_a = any(t.name == a for t in ca_bc.get_terminals())
    candidates = []
    if not ab_has_c:
        candidates.append("ab")
    if not ac_has_b:
        candidates.append("ac")
    if not bc_has_a:
        candidates.append("bc")
    return candidates[0] if len(candidates) == 1 else None  # None if unresolved/polytomy


def orient_triplet_to_tree(tree, a, b, c):
    sp = sister_pair_for_triplet(tree, a, b, c)
    if sp == "ab":
        return a, b, c
    elif sp == "ac":
        return a, c, b
    elif sp == "bc":
        return b, c, a
    else:
        return None  # unresolved/polytomy; you may skip these


def quartet_key(p1: str, p2: str, p3: str, o: str) -> str:
    return f"{p1}|{p2}|{p3}|{o}"


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def append_row(path: str, cols: List[str], row_dict: dict):
    """Append a single row to TSV; create header if file doesn't exist."""
    file_exists = os.path.isfile(path)
    df_row = pd.DataFrame([{c: row_dict.get(c, "") for c in cols}])
    df_row.to_csv(path, sep="\t", index=False, header=not file_exists, mode="a")


# -------------------- MAIN (resumable) --------------------
ensure_dir(output_dir)

# Load inputs
seqs = parse_fasta(alignment_path)
tree = Phylo.read(tree_path, "newick")
clade2taxa = parse_clade_map(clade_map_path)
taxon2clade = {t: c for c, ts in clade2taxa.items() for t in ts}

quartet_definitions = generate_quartets_cross_clades(
    seqs=seqs,
    tree=tree,
    clade2taxa=clade2taxa,
    interest=clades_of_interest,
    out_taxon=outgroup_taxon,
    out_clade=outgroup_clade if outgroup_taxon is None else None,
    require_P3_third=require_P3_third_clade,
    use_all_out=use_all_outgroup_taxa,
    max_p1=max_p1_per_clade,
    max_p2=max_p2_per_clade,
    max_p3=max_p3_per_clade,
    max_pairs_cross=max_pairs_cross_clades
)
if not quartet_definitions:
    sys.exit("No quartets were generated. Check clades_of_interest/outgroup and taxa presence.")

# Columns to write (no BH here since we append per-quartet; you can compute BH in a separate pass)
OUT_COLS = [
    "P1","P2","P3","O","P1_clade","P2_clade","P3_clade","O_clade",
    "sites_D","ABBA","BABA","D","D_SE","D_Z","D_blocks","p_D_norm","p_D_t","p_D_method","p_D_used",
    "sites_f4","f4_mean","f4_SE","f4_Z","f4_blocks","p_f4_norm","p_f4_t","p_f4_method","p_f4_used"
]

# Load already processed quartet keys (if any)
processed_keys = set()
if os.path.isfile(output_file):
    try:
        prev = pd.read_csv(output_file, sep="\t", usecols=["P1", "P2", "P3", "O"])
        for _, r in prev.iterrows():
            processed_keys.add(quartet_key(str(r["P1"]), str(r["P2"]), str(r["P3"]), str(r["O"])))
        print(f"Resuming: found {len(processed_keys)} previously processed quartets in {output_file}")
    except Exception as e:
        print(f"Warning: Could not read existing output for resume check ({e}). Proceeding without skip set.")

# Process and append per quartet (resumable)
n_total = len(quartet_definitions)
n_skipped = 0
n_done = 0

for idx, q in enumerate(quartet_definitions, start=1):
    oriented = orient_triplet_to_tree(tree, q["P1"], q["P2"], q["P3"])
    if oriented is None:
        # Unresolved/polytomy; skip
        continue
    p1, p2, p3 = oriented
    qkey = quartet_key(p1, p2, p3, q["O"])

    if qkey in processed_keys:
        n_skipped += 1
        continue

    # recompute clade labels to match oriented P1,P2,P3
    q_oriented = {
        "P1": p1, "P2": p2, "P3": p3, "O": q["O"],
        "P1_clade": taxon2clade.get(p1, q["P1_clade"]),
        "P2_clade": taxon2clade.get(p2, q["P2_clade"]),
        "P3_clade": taxon2clade.get(p3, q["P3_clade"]),
        "O_clade": q["O_clade"],
    }

    # Compute stats
    res = compute_D_and_f4_for_quartet(seqs, p1, p2, p3, q["O"], block_size=block_size)
    p_D_used = res["p_D_t"] if res["p_D_method"] == "t" else res["p_D_norm"]
    p_f4_used = res["p_f4_t"] if res["p_f4_method"] == "t" else res["p_f4_norm"]

    row = {
        **q_oriented,
        "sites_D": res["sites_D"], "ABBA": res["ABBA"], "BABA": res["BABA"],
        "D": res["D"], "D_SE": res["D_SE"], "D_Z": res["D_Z"], "D_blocks": res["D_blocks"],
        "p_D_norm": res["p_D_norm"], "p_D_t": res["p_D_t"], "p_D_method": res["p_D_method"],
        "p_D_used": p_D_used,
        "sites_f4": res["sites_f4"], "f4_mean": res["f4_mean"], "f4_SE": res["f4_SE"],
        "f4_Z": res["f4_Z"], "f4_blocks": res["f4_blocks"],
        "p_f4_norm": res["p_f4_norm"], "p_f4_t": res["p_f4_t"], "p_f4_method": res["p_f4_method"],
        "p_f4_used": p_f4_used,
    }

    append_row(output_file, OUT_COLS, row)
    processed_keys.add(qkey)
    n_done += 1
    if n_done % 50 == 0:
        print(f"Progress: {idx}/{n_total} quartets processed; {n_done} written, {n_skipped} skipped.")

print(f"Finished. Wrote {n_done} new quartets; skipped {n_skipped} already present.")
print(f"Appended results are in: {output_file}")

# -------------------- MULTIPLE-TESTING CORRECTION --------------------
def benjamini_hochberg(p_values: pd.Series) -> pd.Series:
    """Return Benjamini-Hochberg-adjusted p values while preserving missing values."""
    values = pd.to_numeric(p_values, errors="coerce")
    adjusted = pd.Series(float("nan"), index=values.index, dtype="float64")
    valid = values.dropna()
    if valid.empty:
        return adjusted

    if multipletests is not None:
        adjusted.loc[valid.index] = multipletests(
            valid.to_numpy(), method="fdr_bh"
        )[1]
        return adjusted

    ranked = sorted((float(p), idx) for idx, p in valid.items())
    m = len(ranked)
    running = 1.0
    for rank in range(m, 0, -1):
        p_value, idx = ranked[rank - 1]
        running = min(running, p_value * m / rank)
        adjusted.at[idx] = min(running, 1.0)
    return adjusted


def add_bh_qvalues(input_path: str, output_path: str) -> None:
    """Add BH-adjusted q values for the selected D and f4 p values."""
    results = pd.read_csv(input_path, sep="\t")
    results["q_D_BH"] = benjamini_hochberg(results["p_D_used"])
    results["q_f4_BH"] = benjamini_hochberg(results["p_f4_used"])
    results.to_csv(output_path, sep="\t", index=False)


qvalue_output_file = output_file.replace(".tsv", "_with_q.tsv")
add_bh_qvalues(output_file, qvalue_output_file)
print(f"BH-adjusted results are in: {qvalue_output_file}")

